# Laboratorio: Entanglement Swapping

Simulación completa del protocolo de intercambio de entrelazamiento,
base de los repetidores cuánticos del Internet Cuántico.

**Objetivos:**
- Implementar el circuito de entanglement swapping en Qiskit
- Verificar que los extremos A y B quedan entrelazados tras la medición del Repetidor
- Simular el efecto del ruido en la fidelidad del par entrelazado resultante
- Implementar la purificación de entrelazamiento DEJMPS simplificada

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity, entropy, partial_trace
from qiskit.primitives import StatevectorSampler

print('Qiskit importado correctamente')

## 1. Generación de Pares de Bell

In [ ]:
def bell_pair(qubit_a: int, qubit_b: int, n_total: int) -> QuantumCircuit:
    """Crea el par de Bell |Φ+> entre los qubits qubit_a y qubit_b."""
    qc = QuantumCircuit(n_total)
    qc.h(qubit_a)
    qc.cx(qubit_a, qubit_b)
    return qc

# Verificar que el par de Bell es correcto
qc_bell = bell_pair(0, 1, 2)
sv_bell = Statevector.from_instruction(qc_bell)
print(f'|Φ+⟩ = {sv_bell}')
print(f'Amplitudes: {sv_bell.data}')

# Estado reducido del qubit 0 (debe ser la mezcla máxima)
rho_q0 = partial_trace(DensityMatrix(sv_bell), [1])
E = entropy(rho_q0, base=2)
print(f'Entropía de entrelazamiento: {E:.4f} ebits (debe ser 1.0)')

## 2. Circuito de Entanglement Swapping

In [ ]:
# Sistema: A(q0) --- R1(q1) | R2(q2) --- B(q3)
# El Repetidor tiene q1 y q2
# Después del swapping: A(q0) y B(q3) deben estar entrelazados

def entanglement_swapping_circuit() -> QuantumCircuit:
    """
    Circuito de entanglement swapping:
    1. Crea par A-R1 y par R2-B
    2. El Repetidor realiza medición de Bell en (R1, R2)
    3. Aplica correcciones de Pauli en B según el resultado
    """
    qr = QuantumRegister(4, 'q')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, cr)
    
    # Paso 1: Crear par A-R1 (qubits 0 y 1)
    qc.h(0)
    qc.cx(0, 1)
    
    # Paso 2: Crear par R2-B (qubits 2 y 3)
    qc.h(2)
    qc.cx(2, 3)
    
    qc.barrier()
    
    # Paso 3: Medición de Bell en el Repetidor (qubits 1 y 2)
    qc.cx(1, 2)   # Transformación a la base de Bell
    qc.h(1)
    qc.measure(1, 0)  # Medir R1 → c[0]
    qc.measure(2, 1)  # Medir R2 → c[1]
    
    # Paso 4: Correcciones de Pauli en B (qubit 3)
    # Si c[0]=1: aplicar Z en B
    # Si c[1]=1: aplicar X en B
    with qc.if_test((cr[0], 1)):
        qc.z(3)
    with qc.if_test((cr[1], 1)):
        qc.x(3)
    
    return qc

qc_swap = entanglement_swapping_circuit()
print('Circuito de Entanglement Swapping:')
print(qc_swap.draw(fold=-1))

## 3. Verificación: A y B quedan entrelazados

In [ ]:
# Para verificar sin medición, simulamos el estado antes de medir
# y verificamos que el estado de 4 qubits tiene la estructura correcta

# Circuito sin medición para análisis del estado
qc_analisis = QuantumCircuit(4)

# Crear pares
qc_analisis.h(0); qc_analisis.cx(0, 1)  # Par A-R1
qc_analisis.h(2); qc_analisis.cx(2, 3)  # Par R2-B

# Transformación de Bell en el Repetidor (sin medir)
qc_analisis.cx(1, 2)
qc_analisis.h(1)

sv_4q = Statevector.from_instruction(qc_analisis)
probs = sv_4q.probabilities_dict()

print('Probabilidades del estado de 4 qubits tras la transformación de Bell:')
print('(formato: q3 q2 q1 q0 = B R2 R1 A)')
for state, prob in sorted(probs.items(), key=lambda x: -x[1]):
    if prob > 0.01:
        print(f'  |{state}⟩: {prob:.4f}')

print()
print('Observación: los estados de AB (q3 q0) siempre son 00 o 11,')
print('indicando que A y B están entrelazados independientemente del resultado del Repetidor.')

# Calcular entropía de entrelazamiento de A-B condicionada
# Para cada resultado de la medición del Repetidor
for result_R1R2, label in [('00', '|00⟩_R'), ('01', '|01⟩_R'), ('10', '|10⟩_R'), ('11', '|11⟩_R')]:
    # Proyectar en el subespacio del Repetidor
    sv_data = sv_4q.data.reshape(2, 2, 2, 2)  # [q3, q2, q1, q0]
    r1, r2 = int(result_R1R2[0]), int(result_R1R2[1])  # q1=r1, q2=r2
    proj = sv_data[:, r2, r1, :].flatten()
    norm = np.linalg.norm(proj)
    if norm > 0.01:
        proj_normalized = proj / norm
        dm_AB = DensityMatrix(proj_normalized)
        rho_A = partial_trace(dm_AB, [1])
        E = entropy(rho_A, base=2)
        print(f'  Resultado R={label}: E(A:B) = {E:.4f} ebits, probabilidad = {norm**2:.4f}')

## 4. Impacto del Ruido en la Fidelidad

In [ ]:
def noisy_bell_pair(p_noise: float) -> np.ndarray:
    """
    Genera un par de Bell con canal despolarizante de parámetro p_noise.
    Devuelve la matriz de densidad del par.
    """
    # Estado ideal |Φ+>
    phi_plus = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
    rho_ideal = np.outer(phi_plus, phi_plus.conj())
    
    # Canal despolarizante en el espacio de 2 qubits
    I4 = np.eye(4)
    rho_noisy = (1 - p_noise) * rho_ideal + p_noise / 4 * I4
    return rho_noisy

phi_plus = np.array([1, 0, 0, 1]) / np.sqrt(2)
rho_target = np.outer(phi_plus, phi_plus)

p_values = np.linspace(0, 1, 50)
fidelities = []

for p in p_values:
    rho_noisy = noisy_bell_pair(p)
    F = state_fidelity(rho_target, rho_noisy)
    fidelities.append(F)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p_values, fidelities, color='#e74c3c', linewidth=2)
ax.axhline(y=0.5, color='#8e44ad', linestyle='--', alpha=0.7, label='F=0.5 (umbral purificación)')
ax.set_xlabel('Parámetro de ruido p')
ax.set_ylabel('Fidelidad F(ρ, |Φ+⟩)')
ax.set_title('Fidelidad del par de Bell bajo canal despolarizante')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'Fidelidad con p=0.0 (ideal): {fidelities[0]:.4f}')
print(f'Fidelidad con p=0.5: {fidelities[25]:.4f}')
print(f'Fidelidad con p=1.0 (totalmente mixto): {fidelities[-1]:.4f}')
print('Para purificar, se necesita F > 0.5')

## 5. Purificación de Entrelazamiento (DEJMPS simplificado)

In [ ]:
def dejmps_fidelity(F: float) -> tuple:
    """
    Calcula la fidelidad después de un paso de purificación DEJMPS.
    Asume estado Werner: ρ = F|Φ+><Φ+| + (1-F)/4 * I.
    
    Returns:
        (F_new, p_success): nueva fidelidad y probabilidad de éxito
    """
    epsilon = (1 - F) / 3  # coeficiente de mezcla
    numerator = F**2 + epsilon**2
    denominator = (F + epsilon)**2 - 2*F*epsilon
    # Fórmula exacta DEJMPS para estado Werner
    F_new = (F**2 + epsilon**2) / (F**2 + 2*F*epsilon + 5*epsilon**2)
    p_success = F**2 + 2*F*epsilon + 5*epsilon**2
    return F_new, p_success

# Simular iteraciones de purificación
F0 = 0.75  # fidelidad inicial
F_history = [F0]
p_success_history = [1.0]
overhead_pairs = [1]

F_current = F0
total_pairs = 1

for step in range(10):
    if F_current >= 0.99:
        break
    F_new, p_success = dejmps_fidelity(F_current)
    total_pairs = total_pairs / p_success  # pares iniciales necesarios
    F_current = F_new
    F_history.append(F_current)
    p_success_history.append(p_success)
    overhead_pairs.append(total_pairs)

print(f'Purificación DEJMPS partiendo de F0 = {F0}:')
print(f'{'Paso':<6} {'F':<10} {'p_success':<12} {'Pares necesarios'}')
print('-' * 45)
for i, (F, p, pairs) in enumerate(zip(F_history, p_success_history, overhead_pairs)):
    print(f'{i:<6} {F:<10.6f} {p:<12.6f} {pairs:.2f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(len(F_history)), F_history, 'o-', color='#2ecc71', linewidth=2)
ax1.axhline(y=0.99, color='gray', linestyle='--', alpha=0.5, label='F=0.99')
ax1.set_xlabel('Iteración'); ax1.set_ylabel('Fidelidad'); ax1.legend()
ax1.set_title('Convergencia de la purificación DEJMPS')

ax2.semilogy(range(len(overhead_pairs)), overhead_pairs, 's-', color='#e74c3c', linewidth=2)
ax2.set_xlabel('Iteración'); ax2.set_ylabel('Pares iniciales necesarios (log)')
ax2.set_title('Overhead de pares')

plt.tight_layout()
plt.show()